# Blog 1 Notebook 02: Explore Teacher Traces And SFT Data

This notebook owns the training-data decision. The teacher-generation script produces BAML-canonical SFT trace rows from successful trajectories. Here we inspect them, canonicalize each assistant target to one BAML-style `draft` plus executable `output` decision, choose the sequence-length cutoff, and write the final filtered SFT file used by training scripts.

![Teacher trajectory to SFT rows](../assets/teacher-trajectory-to-sft-sketchnote.png)

In [ ]:
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

from transformers import AutoTokenizer

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "common" / "sql_agent.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common import config as cfg
from common import sql_agent

OUTPUT_DIR = PROJECT_ROOT / "outputs"
SFT_TRACE_ROWS_PATH = OUTPUT_DIR / "gpt_5_5_medium_sql_agent_train_baml_sft_trace_rows.jsonl"
SFT_TRACE_REPORT_PATH = SFT_TRACE_ROWS_PATH.with_suffix(".report.json")
print("BAML-canonical SFT trace rows:", SFT_TRACE_ROWS_PATH)
print("Report:", SFT_TRACE_REPORT_PATH)

## Load BAML-Canonical SFT Trace Rows

In [ ]:
sft_trace_rows = cfg.read_jsonl(SFT_TRACE_ROWS_PATH)
report = json.loads(SFT_TRACE_REPORT_PATH.read_text()) if SFT_TRACE_REPORT_PATH.exists() else {}
summary = report.get("summary", {})

print("BAML-canonical SFT trace rows:", len(sft_trace_rows))
if summary:
    print("Teacher success:", f"{summary.get('success')}/{summary.get('total')}")
    print("Submitted:", summary.get("submitted"))
    print("Parse failures:", summary.get("parse_failures"))
print("First row keys:", list(sft_trace_rows[0].keys()))

## What Did The Teacher Solve?

In [ ]:
rows_by_task = defaultdict(list)
for row in sft_trace_rows:
    rows_by_task[row["task_id"]].append(row)

print("Successful tasks represented:", len(rows_by_task))
print("Rows per task:")
for count, freq in sorted(Counter(len(rows) for rows in rows_by_task.values()).items()):
    print(f"  {count} rows: {freq} tasks")
print("Categories in kept rows:")
for category, count in Counter(row["category"] for row in sft_trace_rows).most_common():
    print(f"  {category}: {count}")

## Inspect One BAML-Canonical Trace Row

The BAML-canonical target may include harness-specific formatting. The canonical target is the parsed teacher draft plus exactly one executable output action.

In [ ]:
example = sft_trace_rows[0]
canonical = sql_agent.canonical_sft_row(example)

print("Row id:", example["id"])
print("Task id:", example["task_id"])
print("Teacher BAML output:")
print(example["teacher_baml_output"][:2000])
print("\nParsed teacher action:")
print(json.dumps(example["teacher_action"], indent=2, ensure_ascii=False))
print("\nCanonical assistant target:")
print(canonical["messages"][-1]["content"])

## One Full Solved Trajectory Becomes Multiple Training Rows

In [ ]:
task_id, task_rows = next(iter(rows_by_task.items()))
print("Task:", task_id)
print("SFT rows extracted:", len(task_rows))
for row in task_rows:
    canonical = sql_agent.canonical_sft_row(row)
    print("\n" + "=" * 80)
    print(row["id"])
    print("Prompt messages before target:", len(canonical["messages"]) - 1)
    print("Target:", canonical["messages"][-1]["content"])

## Token Length Distribution

Sequence length is a training-data decision. We measure it after applying the student tokenizer and the chat template.

In [ ]:
TOKENIZER_MODEL = cfg.MLX_STUDENT_MODEL
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL, trust_remote_code=True)

for max_length in [2048, 2560, 3072, 4096]:
    prepared = sql_agent.prepare_sft_rows(sft_trace_rows, tokenizer, max_length, cfg.SFT_VALIDATION_FRACTION)
    stats = prepared["stats"]
    print(
        f"max_length={max_length:4d} "
        f"kept={stats['kept_rows']:3d}/{stats['source_rows']} "
        f"dropped={stats['dropped_rows']:3d} "
        f"p50={stats.get('p50')} p90={stats.get('p90')} p95={stats.get('p95')} max={stats.get('max')}"
    )

## Prompt Tokens vs Target Tokens

Only assistant target tokens should contribute to the SFT loss. The prompt/history provides state; the target is the next teacher decision: short draft plus executable output action.

In [ ]:
canonical = sql_agent.canonical_sft_row(example)
prompt_text = tokenizer.apply_chat_template(canonical["messages"][:-1], tokenize=False, add_generation_prompt=True, enable_thinking=cfg.QWEN_ENABLE_THINKING)
full_text = tokenizer.apply_chat_template(canonical["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=cfg.QWEN_ENABLE_THINKING)
prompt_tokens = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
full_tokens = tokenizer(full_text, add_special_tokens=False)["input_ids"]

print("Prompt tokens:", len(prompt_tokens))
print("Target tokens:", len(full_tokens) - len(prompt_tokens))
print("Full SFT row tokens:", len(full_tokens))
print("\nTarget text:")
print(canonical["messages"][-1]["content"])

## Write Final Filtered SFT Data

This is the file the training scripts should consume. The choice is visible here instead of hidden inside training.

In [ ]:
MAX_SEQ_LENGTH = 3072
FINAL_SFT_PATH = OUTPUT_DIR / f"gpt_5_5_medium_sql_agent_train_sft_canonical_{MAX_SEQ_LENGTH}.jsonl"
FINAL_REPORT_PATH = FINAL_SFT_PATH.with_suffix(".report.json")

prepared = sql_agent.prepare_sft_rows(sft_trace_rows, tokenizer, MAX_SEQ_LENGTH, cfg.SFT_VALIDATION_FRACTION)
final_rows = prepared["rows"]
report_payload = {
    "source": str(SFT_TRACE_ROWS_PATH),
    "tokenizer": TOKENIZER_MODEL,
    "max_seq_length": MAX_SEQ_LENGTH,
    "stats": prepared["stats"],
}

cfg.write_jsonl(FINAL_SFT_PATH, final_rows)
cfg.write_json(FINAL_REPORT_PATH, report_payload)

print("Wrote final SFT rows:", FINAL_SFT_PATH)
print("Wrote report:", FINAL_REPORT_PATH)
print(json.dumps(prepared["stats"], indent=2))

## Training Command

After this notebook writes the final file, train with the NVIDIA/Unsloth path:

In [ ]:
print(f"""# Mac / Apple Silicon
uv run python 1-distilling-a-0-8b-tool-calling-agent/train_unsloth.py \
  --backend mlx \
  --model {cfg.MLX_STUDENT_MODEL} \
  --train-path {FINAL_SFT_PATH.relative_to(PROJECT_ROOT)} \
  --output-dir outputs/qwen3_5_0_8b_mlx_tune_sql_agent_gpt_teacher_sft_{MAX_SEQ_LENGTH}

# NVIDIA / CUDA
uv run python 1-distilling-a-0-8b-tool-calling-agent/train_unsloth.py \
  --backend cuda \
  --model {cfg.UNSLOTH_STUDENT_MODEL} \
  --train-path {FINAL_SFT_PATH.relative_to(PROJECT_ROOT)} \
  --output-dir outputs/qwen3_5_0_8b_unsloth_sql_agent_gpt_teacher_sft_{MAX_SEQ_LENGTH}""")
